In [1]:
import numpy as np
import pandas as pd
import os
import h5py
import sys

from tqdm import tqdm

import torch

sys.path.insert(0, '../src/')
import phiseg.phiseg_pl as phiseg

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
path_to_data_test     = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/TEST/'
path_to_inference_out = '/home/paul/Desktop/data_ped_unc/predictions/phiseg/'

# Number of stochastic samples drawn from the PHiSeg prior per image
n_samples = 20

os.makedirs(path_to_inference_out, exist_ok=True)

In [3]:
def load_phiseg_model(ckpt_path):
    input_channels = 1
    num_classes    = 2
    num_filters    = [32, 64, 128, 192, 192, 192, 192]
    model = phiseg.PHISeg(input_channels, num_classes, num_filters)

    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt.get("state_dict", ckpt)

    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("model."): k = k[len("model."):]
        if k.startswith("net."):   k = k[len("net."):]
        new_state_dict[k] = v

    model.load_state_dict(new_state_dict)
    model.eval()
    return model

In [4]:
# Fill in checkpoint path before running
model_path = '/home/paul/Desktop/data_ped_unc/models_eval/phiseg/comb/version_0/checkpoints/epoch=178-step=43676.ckpt'

print(f"Loading PHiSeg model from {model_path} ...")
model = load_phiseg_model(model_path).to(device)
print(f"Model loaded. Will draw {n_samples} samples per image.")

Loading PHiSeg model from /home/paul/Desktop/data_ped_unc/models_eval/phiseg/comb/version_0/checkpoints/epoch=178-step=43676.ckpt ...
Model loaded. Will draw 20 samples per image.


In [5]:
file_names = sorted(os.listdir(path_to_data_test))
print(f"Running inference on {len(file_names)} files...")

for file in tqdm(file_names):
    full_filepath = os.path.join(path_to_data_test, file)
    out_filepath  = os.path.join(path_to_inference_out, file)

    with h5py.File(full_filepath, 'r') as src, h5py.File(out_filepath, 'w') as dst:
        phases_present = [p for p in ['ed', 'es'] if p in src]

        for phase in phases_present:
            img = src[phase]['image'][()]  # (H, W)
            img_tensor = torch.Tensor(img).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, H, W)

            # PHiSeg stochastic inference: returns softmax samples (n_samples, n_classes, H, W)
            with torch.no_grad():
                softmax_samples = model.sample_softmax(img_tensor, n_samples=n_samples)

            logits_arr = softmax_samples.cpu().numpy()  # (n_samples, n_classes, H, W)

            grp = dst.require_group(phase)
            grp.create_dataset('logits', data=logits_arr)

print("Done. Outputs saved to", path_to_inference_out)

Running inference on 314 files...


100%|██████████| 314/314 [00:24<00:00, 12.79it/s]

Done. Outputs saved to /home/paul/Desktop/data_ped_unc/predictions/phiseg/
